## 세포라 top10 수집및 모든리뷰 2년치 수집코드

In [1]:
import requests
import json
import time
import random
from datetime import datetime, timedelta

# ── 설정 ──────────────────────────────────────────────────────────
BV_PASSKEY       = "calXm2DyQVjcCy9agq85vmTJv5ELuuBCF2sdg4BnJzJus"
BV_API_URL       = "https://api.bazaarvoice.com/data/reviews.json"
RANK_SAVE_FILE   = "sephora_rankings_current.jsonl"   
REVIEW_SAVE_FILE = "sephora_reviews_master.jsonl"   

# [날짜 설정] 현재 날짜 기준 정확히 2년 전 날짜 계산
CUTOFF_DATE = datetime.now() - timedelta(days=365 * 2)

SEPHORA_API_URL = (
    "https://www.sephora.com/api/v2/catalog/categories/skincare/seo"
    "?targetSearchEngine=NLP"
    "&sortBy=P_BEST_SELLING%3A1%3A%3AP_RATING%3A1%3A%3AP_PROD_NAME%3A0"
    "&currentPage=1&pageSize=60&content=true"
    "&includeRegionsMap=true&pickupRampup=true&sddRampup=true"
    "&includeEDD=true&loc=en-US&ch=rwd&user-segment=external-app"
)
HEADERS = {
    "User-Agent"     : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept"         : "application/json",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer"        : "https://www.sephora.com/shop/skincare?sortBy=BEST_SELLING",
    "Origin"         : "https://www.sephora.com",
}


# ── 함수: 단일 상품 리뷰 수집 (최근 2년치 필터링) ───────────────────
def get_sephora_reviews_2years(product_id, product_name):
    all_reviews = []
    limit  = 100
    offset = 0
    total  = None
    
    # 루프 중단을 위한 플래그
    stop_collecting = False

    print(f"\n  🚀 [{product_name[:35]}] 리뷰 수집 (기준: {CUTOFF_DATE.strftime('%Y-%m-%d')} 이후)")

    while not stop_collecting:
        params = {
            "Filter"    : ["contentlocale:en*", f"ProductId:{product_id}"],
            "Sort"      : "SubmissionTime:desc",  # 최신순 정렬 필수
            "Limit"     : limit,
            "Offset"    : offset,
            "Include"   : "Products,Comments",
            "Stats"     : "Reviews",
            "passkey"   : BV_PASSKEY,
            "apiversion": "5.4",
            "Locale"    : "en_US",
        }
        try:
            response = requests.get(BV_API_URL, params=params, timeout=15)
            data     = response.json()

            if total is None:
                total = data.get("TotalResults", 0)
                print(f"  📊 상품 전체 리뷰 수: {total}개")

            results = data.get("Results", [])
            if not results:
                break

            for rev in results:
                # 1. 날짜 파싱 (예: 2026-03-14T18:33:38.000+00:00)
                sub_time_str = rev.get("SubmissionTime")
                if sub_time_str:
                    # ISO 형식 문자열을 datetime 객체로 변환 (앞 19자리 'YYYY-MM-DDTHH:MM:SS'만 사용)
                    sub_time = datetime.strptime(sub_time_str[:19], "%Y-%m-%dT%H:%M:%S")
                    
                    # 2. 날짜 비교: 기준일(2년 전)보다 과거이면 즉시 중단
                    if sub_time < CUTOFF_DATE:
                        stop_collecting = True
                        break
                
                # 3. 데이터 저장
                all_reviews.append({
                    "product_id"    : product_id,
                    "ReviewId"      : rev.get("Id"),
                    "Author"        : rev.get("UserNickname"),
                    "Rating"        : rev.get("Rating"),
                    "Title"         : rev.get("Title"),
                    "ReviewText"    : rev.get("ReviewText"),
                    "SubmissionTime": sub_time_str,
                    "Date"          : sub_time.strftime("%Y-%m-%d") if sub_time_str else "N/A",
                    "IsRecommended" : rev.get("IsRecommended"),
                    "SkinType"      : rev.get("ContextDataValues", {}).get("skinType", {}).get("ValueLabel", "N/A"),
                    "AgeRange"      : rev.get("ContextDataValues", {}).get("ageRange", {}).get("ValueLabel", "N/A"),
                    "Incentivized"  : rev.get("ContextDataValues", {}).get("IncentivizedReview", {}).get("ValueLabel", "N/A")
                })

            if stop_collecting:
                print(f"\n  📍 2년 이전 리뷰 도달 - 수집 중단")
                break

            offset += limit
            print(f"  🔄 분석 중... 현재 {len(all_reviews)}개 확보", end="\r")
            time.sleep(random.uniform(0.3, 0.6))

            # API 오프셋이 전체 개수를 넘으면 종료
            if offset >= total:
                break

        except Exception as e:
            print(f"\n  ❌ 에러 발생: {e}")
            break

    print(f"\n  ✨ 최종 수집 완료: {len(all_reviews)}개")
    return all_reviews


# ── STEP 1: 세포라 스킨케어 베스트셀러 Top 10 수집 ────────────────
print(f"📡 Sephora API 호출 (기준일: {CUTOFF_DATE.strftime('%Y-%m-%d')})")
resp = requests.get(SEPHORA_API_URL, headers=HEADERS, timeout=20)
resp.raise_for_status()
data = resp.json()

products_raw = data.get("products") or data.get("catalog", {}).get("products") or []

if not products_raw:
    print("⚠️ 상품 목록을 찾지 못했습니다.")
else:
    rank_data_list    = []
    all_review_master = []
    rank_count        = 1

    for product in products_raw:
        if rank_count > 10:
            break

        # 광고/스폰서 상품 제외
        if product.get("isSponsored") or product.get("sponsored") or product.get("adBadge"):
            continue

        brand       = product.get("brandName", "N/A")
        title       = product.get("displayName", "N/A")
        sku         = product.get("currentSku", {})
        price       = sku.get("listPrice") or product.get("listPrice") or "N/A"
        rating      = float(product.get("rating", 0) or 0)
        reviews_cnt = int(product.get("reviews", 0) or 0)
        product_id  = product.get("productId", "N/A")
        url_path    = product.get("targetUrl") or product.get("url", "")
        product_url = f"https://www.sephora.com{url_path}" if url_path and not url_path.startswith("http") else url_path

        rank_data_list.append({
            "rank"         : rank_count,
            "brand"        : brand,
            "title"        : title,
            "rating"       : rating,
            "reviews"      : reviews_cnt,
            "price"        : price,
            "url"          : product_url,
            "product_id"   : product_id,
            "platform"     : "Sephora",
            "collected_at" : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        })
        print(f"\n📍 {rank_count}위: [{brand}] {title[:35]}")

        # ── STEP 2: 최근 2년치 리뷰 수집 실행 ─────────────────────────────
        reviews = get_sephora_reviews_2years(product_id, title)
        all_review_master.extend(reviews)

        rank_count += 1

    # ── STEP 3: JSONL 저장 ───────────────────────────────────────
    with open(RANK_SAVE_FILE, "w", encoding="utf-8") as f:
        for entry in rank_data_list:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    with open(REVIEW_SAVE_FILE, "w", encoding="utf-8") as f:
        for review in all_review_master:
            f.write(json.dumps(review, ensure_ascii=False) + "\n")

    print(f"\n📊 작업 완료! 상품 {len(rank_data_list)}개 / 총 리뷰 {len(all_review_master)}개 저장됨")

📡 Sephora API 호출 (기준일: 2024-03-21)

📍 1위: [rhode] Glazing Milk Ceramide Facial Essenc

  🚀 [Glazing Milk Ceramide Facial Essenc] 리뷰 수집 (기준: 2024-03-21 이후)
  📊 상품 전체 리뷰 수: 1945개
  🔄 분석 중... 현재 1945개 확보
  ✨ 최종 수집 완료: 1945개

📍 2위: [rhode] Peptide Lip Tint Nourishing Glaze

  🚀 [Peptide Lip Tint Nourishing Glaze] 리뷰 수집 (기준: 2024-03-21 이후)
  📊 상품 전체 리뷰 수: 2179개


KeyboardInterrupt: 

# 세포라 번역코드

In [4]:
import json
import time
import re
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from deep_translator import GoogleTranslator

# ── 설정 ──────────────────────────────────────────────────────────
INPUT_FILE  = "./sephora_reviews_master.jsonl"
OUTPUT_FILE = "sephora_master_translated_en_ko.jsonl"

BODY_COL    = "ReviewText"  # 세포라 리뷰 본문 키
ITEM_ID_COL = "product_id"
RATING_COL  = "Rating"

N_SAMPLE    = None  # 테스트 시 숫자 입력, 전체 번역 시 None
CHUNK_SIZE  = 5     # 한 번에 묶어서 번역할 개수
MAX_WORKERS = 4     # 병렬 스레드 수 (IP 차단 위험 시 2로 낮춤)
MAX_RETRIES = 3
RETRY_SLEEP = 2.0
CHUNK_DELAY = 0.3   # 영어-한국어는 속도가 빨라 딜레이를 약간 줄임

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1. 유틸리티 로직 (번호 매기기 방식)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_numbered(texts: list) -> str:
    """리뷰 리스트를 [1] 문장1 \n [2] 문장2 형태로 조립"""
    return "\n".join(f"[{i+1}] {str(t).strip()}" for i, t in enumerate(texts))

def parse_numbered(text: str, expected_n: int) -> list:
    """번역된 텍스트에서 [1], [2] 패턴을 찾아 리스트로 분리"""
    pattern = re.compile(r'\[(\d+)\]\s*(.*?)(?=\[\d+\]|$)', re.DOTALL)
    found = pattern.findall(text)
    result = {int(idx): body.strip() for idx, body in found}
    return [result.get(i + 1, "") for i in range(expected_n)]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2. 번역 엔진
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def translate_single(text: str, src: str, tgt: str) -> str:
    """청크 실패 시 사용하는 개별 번역 함수"""
    if not text or not str(text).strip(): return ""
    for attempt in range(MAX_RETRIES):
        try:
            res = GoogleTranslator(source=src, target=tgt).translate(str(text))
            if res: return res.strip()
        except:
            time.sleep(RETRY_SLEEP * (attempt + 1))
    return "번역실패"

def translate_chunk_en_ko(texts: list) -> list:
    """영어 -> 한국어 청크 번역 실행"""
    if not texts: return []
    joined = build_numbered(texts)
    
    for attempt in range(MAX_RETRIES):
        try:
            result = GoogleTranslator(source="en", target="ko").translate(joined)
            if not result: raise ValueError("빈 응답")
            
            parts = parse_numbered(result, len(texts))
            # 모든 파트가 정상적으로 분리되었는지 확인
            if all(p.strip() for p in parts):
                return parts
            
            # 파싱 실패 시 단건 번역으로 보충
            for i, p in enumerate(parts):
                if not p: parts[i] = translate_single(texts[i], "en", "ko")
            return parts
        except:
            time.sleep(RETRY_SLEEP * (attempt + 1))
            
    return [translate_single(t, "en", "ko") for t in texts]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3. 메인 파이프라인
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def main():
    print(f"📥 데이터 로드 중: {INPUT_FILE}")
    records = []
    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line))
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없습니다: {INPUT_FILE}")
        return

    df = pd.DataFrame(records)
    if N_SAMPLE: df = df.head(N_SAMPLE)
    
    total_count = len(df)
    print(f"✅ 총 {total_count:,}건 로드 완료")

    # 데이터 준비
    texts = df[BODY_COL].fillna("").astype(str).tolist()
    chunks = [texts[i:i+CHUNK_SIZE] for i in range(0, total_count, CHUNK_SIZE)]
    n_chunks = len(chunks)

    print(f"🚀 병렬 번역 시작 (스레드={MAX_WORKERS}, 청크={CHUNK_SIZE})")
    start_time = time.time()

    def run_chunk(chunk):
        res = translate_chunk_en_ko(chunk)
        time.sleep(CHUNK_DELAY)
        return res

    all_ko = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # tqdm 진행바와 함께 병렬 실행
        chunk_results = list(tqdm(
            executor.map(run_chunk, chunks),
            total=n_chunks,
            desc="Sephora 번역 중",
            unit="chunk"
        ))

    # 결과 병합
    for chunk in chunk_results:
        all_ko.extend(chunk)

    df['ReviewText_ko'] = all_ko[:total_count]
    elapsed = time.time() - start_time

    # 통계 및 저장
    success_rate = (df['ReviewText_ko'] != "번역실패").mean() * 100
    print(f"\n⏱ 번역 완료: {elapsed/60:.2f}분 소요 (성공률: {success_rate:.1f}%)")

    print(f"💾 결과 저장 중: {OUTPUT_FILE}")
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for record in df.to_dict(orient='records'):
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

    # 샘플 출력
    print("\n📋 번역 샘플 (최근 3건):")
    for _, row in df.head(3).iterrows():
        print(f"  ★{row[RATING_COL]} | 원문: {str(row[BODY_COL])[:50]}...")
        print(f"        | 번역: {str(row['ReviewText_ko'])[:50]}...")
        print("-" * 50)

if __name__ == "__main__":
    main()

📥 데이터 로드 중: ./sephora_reviews_master.jsonl
✅ 총 13,727건 로드 완료
🚀 병렬 번역 시작 (스레드=4, 청크=5)


Sephora 번역 중: 100%|██████████| 2746/2746 [17:55<00:00,  2.55chunk/s]



⏱ 번역 완료: 17.92분 소요 (성공률: 100.0%)
💾 결과 저장 중: sephora_master_translated_en_ko.jsonl

📋 번역 샘플 (최근 3건):
  ★5 | 원문: It’s so good and moisturizing that sometimes I use...
        | 번역: 보습력도 좋고 너무 좋아서 가끔 보습제로 사용하고 있어요. 다른 것은 필요하지 않습니다. ...
--------------------------------------------------
  ★5 | 원문: I’m genuinely OBSESSED with this glazing milk 🤍 Th...
        | 번역: 저는 이 글레이징 밀크에 진심으로 빠져있습니다 🤍 질감이 전부예요. 너무 가볍고 부드러우며...
--------------------------------------------------
  ★1 | 원문: I had such high hopes for this product and was ver...
        | 번역: 저는 이 제품에 대해 큰 기대를 갖고 있었는데 결과에 매우 실망했습니다. 메이크업 전 베이...
--------------------------------------------------


# 세포라 kebert 1차 분류

In [ ]:
import re
import json
import pandas as pd
from tqdm import tqdm
from keybert import KeyBERT

# =========================
# 설정
# =========================
INPUT_FILE   = "./sephora_master_translated_en_ko.jsonl"
OUTPUT_CSV   = "./sephora_keybert_en_categorized.csv"
OUTPUT_JSONL = "./sephora_keybert_en_categorized.jsonl"

TEXT_COL       = "ReviewText"
TOP_N_KEYWORDS = 5

CATEGORY_KEYWORDS = {
    "효과_성분": [
        "moistur", "hydrat", "hydrating", "whitening", "brighten", "elastic", "firm",
        "wrinkle", "anti-aging", "pore", "glow", "radian", "regenerat",
        "sooth", "calm", "antioxidant", "absorb", "penetrat", "tone", "improve",
        "vitamin", "retinol", "hyaluronic", "ceramide", "niacinamide", "peptid", "vegan",
        "plump", "clear", "even", "spot", "pigment", "aha", "bha", "acid", "exfoliat",
        # 추가: 피부타입 맥락
        "oily skin", "combination skin", "sensitive skin", "works for my skin",
        "dry skin type", "acne-prone skin",
        # 추가: 결과/시간 표현
        "result", "noticeabl", "overnight", "immediately", "after using",
        "after one week", "after a month", "after two week",
        # 추가: 사용 맥락
        "routine", "layer", "morning", "night cream",
        # 추가: 선케어 (제품군 해당 시)
        "spf", "sunscreen", "uv", "sun protect", "reef safe",
    ],
    "사용감_텍스처": [
        "appl", "blend", "texture", "consistenc", "watery", "runny", "thick", "viscos",
        "stick", "tacky", "fresh", "light", "weightless", "heavy", "soft", "smooth",
        "stiff", "greasy", "pill", "flake", "feel", "finish", "rub",
        "oily", "matte", "dewy", "sink", "patchy", "chalky", "white cast",
        # 추가: 메이크업 베이스/선케어 관련
        "pore-filling", "pore filling", "blur", "setting", "blot", "primer",
        "spread", "glide", "pack", "apply thin", "build up",
    ],
    "향_냄새": [
        "scent", "smell", "fragranc", "unscented", "fragrance-free", "odor",
        "subtle", "mild", "strong", "overpowering", "artificial", "natural", "perfume",
        "stink", "aroma", "nose",
        # 추가
        "whiff", "chemical smell", "medicin", "floral", "citrus",
    ],
    "피부_트러블_부작용": [
        "trouble", "breakout", "pimple", "acne", "irritat", "sting", "burn", "itch",
        "red", "redness", "peel", "tight", "sensitiv", "allerg", "dermatitis",
        "reaction", "side effect", "break out", "rash", "harsh",
        "drying", "dried out", "flaky", "dry patch",
        "clog", "purg", "cyst", "bump",
        # 추가: 자극 표현
        "tingle", "sting", "inflam", "swell", "hive", "welt",
        "made my skin worse", "broke me out", "not agree",
    ],
    "포장_배송": [
        "packag", "box", "bottle", "container", "case", "pump", "tube", "ship",
        "deliver", "late", "slow", "arriv", "damag", "broken",
        "leak", "spill", "wrap",
        "fast ship", "arrived fast", "quick deliver",
        "dropper", "cap", "lid", "spray", "nozzle", "shipped", "unseal",
        # 추가
        "packaging", "travel size", "full size", "well-packaged", "poorly packaged",
        "dent", "crush", "tamper",
    ],
    "가격_가성비": [
        "price", "cost", "valu", "expensiv", "pricy", "cheap", "afford", "reasonabl",
        "sale", "discount", "coupon", "buck", "money", "worth", "deal", "size", "amount",
        "pricey", "bargain", "rip off", "waste",
        # 추가: 용량 관련 가성비 표현
        "goes a long way", "a little goes", "last a long time", "last me",
        "small amount", "tiny bit", "lasts forever", "run out fast", "finish quickly",
    ],
    "고객서비스": [
        "custom", "service", "support", "respond", "response", "refund", "return",
        "exchang", "complain", "inquir", "answer", "contact", "issue",
        # 추가
        "seller", "vendor", "representative", "chat", "email them", "called",
        "waited", "resolve", "compensat",
    ],
    "제품불량": [
        "defect", "defective", "faulty", "bug", "dirt", "contaminat", "spoil",
        "weird", "fake", "counterfeit", "knockoff", "differ", "mold", "trash",
        "expir", "rancid", "separat", "empty",
        # 추가
        "smell off", "color off", "look different", "wrong product", "not what",
        "different from", "old stock", "bad batch",
    ],
    "재구매_추천": [
        "repurchas", "buy again", "reorder", "recommend", "holy grail", "staple",
        "go-to", "favorit", "gift", "friend", "keep us", "definitely",
        "love it",
        "hg", "restock", "10/10", "must have", "obsessed",
        # 추가: 비교/전환 표현
        "better than", "switch from", "switch to", "compared to", "used to use",
        "replace", "converted",
        # 추가: 일반 강한 긍정 (내용 없는 짧은 리뷰 흡수)
        "highly recommend", "great product", "works well", "works great",
        "amazing product", "excellent", "perfect product", "love this",
        "love how", "love that", "so good", "so happy",
    ],
    
     "부정_리뷰": [
        # 추가: 부정 추천 표현
        "disappoint", "not recommend", "waste of money", "regret buying",
    ],

    "커버력_색상": [
        "cover", "coverage", "color", "shade", "tone", "tint", "pigment",
        "bright", "dark", "ashy", "oxidiz", "orang", "yellow", "match", "pale",
        "undertone", "fair", "sheer", "opaque", "swatch",
        # 추가
        "full coverage", "medium coverage", "buildable", "natural finish",
        "foundation", "concealer", "bb cream", "cc cream", "tinted",
        "skin tone", "complexion", "too light", "too dark", "perfect match",
    ],
    "지속력_밀착력": [
        "last", "lasting", "longevity", "stay", "adher", "crease", "melt", "fade",
        "long-lasting", "all day", "wear", "hold", "slip", "smudg", "transfer",
        "rub off", "budge", "separate",
        # 추가: 환경 내구성
        "sweat", "sweatproof", "waterproof", "water resistant", "humid",
        "through the day", "by noon", "by midday", "hours later",
        "8 hour", "12 hour", "24 hour",
    ],
}

# =========================
# 유틸
# =========================
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

def clean_light(text):
    text = str(text).replace("\n", " ").replace("\r", " ")
    return re.sub(r"\s+", " ", text).strip()

def extract_keywords(text, top_n=5):
    text = str(text).strip()
    if not text:
        return []
    try:
        kws = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 2),
            stop_words="english",
            top_n=top_n,
            use_mmr=True,
            diversity=0.5
        )
        return [kw for kw, _ in kws]
    except Exception:
        return []

def rule_classify(keywords: list, text: str) -> tuple:
    combined = " ".join(keywords).lower() + " " + text.lower()
    scores = {}
    for cat, kw_list in CATEGORY_KEYWORDS.items():
        count = sum(1 for kw in kw_list if kw in combined)
        if count > 0:
            scores[cat] = count

    if not scores:
        return "unclassified", ["unclassified"]

    sorted_cats = sorted(scores, key=scores.get, reverse=True)
    return sorted_cats[0], sorted_cats[:3]

# =========================
# 모델 로드
# =========================
kw_model = KeyBERT("all-MiniLM-L6-v2")

# =========================
# 데이터 로드
# =========================
df = load_jsonl(INPUT_FILE)
df = df.dropna(subset=[TEXT_COL]).copy()
df = df[df[TEXT_COL].astype(str).str.strip() != ""].reset_index(drop=True)
df["text_for_model"] = df[TEXT_COL].apply(clean_light)
print(f"유효 데이터: {len(df):,}건")

# =========================
# KeyBERT 키워드 추출 + 규칙 분류
# =========================
keywords_list      = []
primary_categories = []
categories_list    = []

for text in tqdm(df["text_for_model"], desc="KeyBERT 분류 (EN)"):
    kws           = extract_keywords(text, top_n=TOP_N_KEYWORDS)
    primary, cats = rule_classify(kws, text)
    keywords_list.append(kws)
    primary_categories.append(primary)
    categories_list.append(cats)

df["keybert_keywords"]  = keywords_list
df["primary_category"]  = primary_categories
df["categories"]        = categories_list

# =========================
# 결과 출력
# =========================
total        = len(df)
classified   = (df["primary_category"] != "unclassified").sum()
unclassified = (df["primary_category"] == "unclassified").sum()

print(f"\n분류 완료: {classified:,}건 ({classified/total*100:.1f}%)")
print(f"미분류:    {unclassified:,}건 ({unclassified/total*100:.1f}%)")
print("\n카테고리 분포:")
print(df["primary_category"].value_counts())

# =========================
# 저장
# =========================
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

print(f"\nCSV:   {OUTPUT_CSV}")
print(f"JSONL: {OUTPUT_JSONL}")

print("\n샘플:")
print(df[[TEXT_COL, "keybert_keywords", "primary_category", "categories"]].head(10))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


총 데이터 수: 13727
컬럼: ['product_id', 'ReviewId', 'Author', 'Rating', 'Title', 'ReviewText', 'SubmissionTime', 'Date', 'IsRecommended', 'SkinType', 'AgeRange', 'Incentivized', 'ReviewText_ko']
유효 데이터 수: 13703


GPT 배치 분류:   2%|▏         | 209/13703 [03:50<4:07:43,  1.10s/it]


KeyboardInterrupt: 

# 세포라 gpt 2차 분류

In [ ]:
import json
import re
from openai import OpenAI

# ── 설정 ──────────────────────────────────────────────────────────────
INPUT_JSONL  = "./sephora_keybert_en_categorized.jsonl"  # KeyBERT 결과 JSONL
OUTPUT_JSONL = "./sephora_final_categorized.jsonl"
BATCH_SIZE   = 30
TEXT_COL     = "ReviewText"

CATEGORIES = [
    "효과_성분", "사용감_텍스처", "향_냄새", "피부_트러블_부작용","부정_리뷰",
    "포장_배송", "가격_가성비", "고객서비스", "제품불량",
    "재구매_추천", "커버력_색상", "지속력_밀착력", "미분류"
]

client = OpenAI()

# ── GPT 배치 분류 ──────────────────────────────────────────────────────
def gpt_classify_batch(batch: list[dict]) -> dict:
    """batch: [{"idx": i, "keywords": [...], "text": "..."}]
    반환: {idx: {"primary_category": ..., "categories": [...]}}
    """
    prompt_items = "\n".join(
        f"[{item['idx']}] keywords={item['keywords']} | text={item['text'][:300]}"
        for item in batch
    )
    system_msg = f"""뷰티 제품 리뷰를 아래 카테고리 중 하나로 분류하세요.
카테고리: {CATEGORIES}

각 리뷰에 대해 JSON 배열로 응답하세요:
[{{"idx": 번호, "primary_category": "카테고리명", "categories": ["카테고리1", ...]}}]
primary_category는 가장 핵심 카테고리 1개, categories는 해당되는 카테고리 모두."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": prompt_items}
        ],
        temperature=0
    )

    raw = response.choices[0].message.content
    # JSON 배열 파싱
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    if not match:
        return {}
    results = json.loads(match.group())
    return {r["idx"]: r for r in results}

# ── 데이터 로드 ────────────────────────────────────────────────────────
records = []
with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

classified   = [r for r in records if r.get("primary_category") != "unclassified"]
unclassified = [r for r in records if r.get("primary_category") == "unclassified"]

print(f"전체: {len(records)} | 분류완료: {len(classified)} | GPT 재분류 대상: {len(unclassified)}")

# ── GPT 배치 실행 ──────────────────────────────────────────────────────
idx_to_record = {i: rec for i, rec in enumerate(unclassified)}
batches = [
    [
        {
            "idx": i,
            "keywords": rec.get("keybert_keywords", []),
            "text": str(rec.get(TEXT_COL, ""))[:300]
        }
        for i, rec in list(idx_to_record.items())[start:start+BATCH_SIZE]
    ]
    for start in range(0, len(unclassified), BATCH_SIZE)
]

print(f"배치 수: {len(batches)} ({BATCH_SIZE}건씩)")

for b_idx, batch in enumerate(batches):
    results = gpt_classify_batch(batch)
    for item in batch:
        i = item["idx"]
        if i in results:
            idx_to_record[i]["primary_category"] = results[i]["primary_category"]
            idx_to_record[i]["categories"]       = results[i]["categories"]
        else:
            idx_to_record[i]["primary_category"] = "이분류"
            idx_to_record[i]["categories"]       = []
    print(f"  배치 {b_idx+1}/{len(batches)} 완료")

# ── 결과 저장 ──────────────────────────────────────────────────────────
final_records = classified + list(idx_to_record.values())

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for rec in final_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"\n저장 완료: {OUTPUT_JSONL} ({len(final_records)}건)")

# ── 분류 결과 확인 ─────────────────────────────────────────────────────
from collections import Counter
cats = [r["primary_category"] for r in final_records]
for cat, cnt in Counter(cats).most_common():
    print(f"  {cat}: {cnt}")